# Project objective
This notebook builds an insurance domain support assistant using local PDF documents, embeddings, and retrieval-augmented generation.


In [3]:
# Import required libraries
# Install packages in the active notebook environment if needed:
# %pip install python-dotenv langchain langchain-openai langchain-community faiss-cpu pymupdf pypdf langchain-text-splitters

import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is missing. Add it to a .env file or your environment before running the notebook.")

print("Environment configured successfully. OpenAI API key detected.")


Environment configured successfully. OpenAI API key detected.


# Environment setup
This cell loads the OpenAI API key and checks that the environment is ready before the RAG workflow starts.


In [4]:
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


DOCS_DIR = r"D:\Technicals\Emeritus\Assignment-3\Assignments-15\documents"
INDEX_PATH = Path(r"D:\Technicals\Emeritus\Assignment-3\Assignments-15\insurance_vectorstore_large")
EMBEDDING_MODEL = "text-embedding-3-large"


def load_pdf_documents(folder_path: str):
    folder = Path(folder_path)
    docs = []
    pdf_files = sorted(folder.glob("*.pdf"))

    if not pdf_files:
        raise FileNotFoundError(f"No PDF files found in {folder}")

    for pdf_file in pdf_files:
        try:
            try:
                loader = PyMuPDFLoader(str(pdf_file))
                file_docs = loader.load()
            except Exception:
                loader = PyPDFLoader(str(pdf_file))
                file_docs = loader.load()

            for doc in file_docs:
                doc.metadata["source"] = pdf_file.name
                if doc.page_content.strip():
                    docs.append(doc)

            print(f"Loaded {len(file_docs)} pages from {pdf_file.name}")
        except Exception as exc:
            print(f"Skipping {pdf_file.name}: {exc}")

    return docs


def build_vector_store(doc_dir: str, persist_path: Path):
    persist_path.mkdir(parents=True, exist_ok=True)
    raw_docs = load_pdf_documents(doc_dir)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=100,
        separators=["\n\n", "\n", ". ", " "],
    )
    chunks = text_splitter.split_documents(raw_docs)
    print(f"Created {len(chunks)} chunks from {len(raw_docs)} pages.")

    embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(str(persist_path))
    print(f"Saved vector store to {persist_path}")
    return vectorstore


index_files = [INDEX_PATH / "index.faiss", INDEX_PATH / "index.pkl"]
if all(file_path.exists() for file_path in index_files):
    embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
    vectorstore = FAISS.load_local(str(INDEX_PATH), embeddings, allow_dangerous_deserialization=True)
    print("Loaded existing FAISS index from disk.")
else:
    print("Vector store not found or stale. Building a fresh index for the current embedding model...")
    vectorstore = build_vector_store(DOCS_DIR, INDEX_PATH)
    print("Built new FAISS index from the PDFs.")


C:\Users\PrabuK\AppData\Local\Temp\ipykernel_19096\3945663402.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader


Loaded existing FAISS index from disk.


# Vector store creation
This cell reads the local PDFs, splits them into chunks, and creates the FAISS vector store used for retrieval.


In [6]:

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
conversation_history = []

def expand_query(question: str) -> str:
    # Basic synonym expansion for insurance-related queries
    expansions = {
        "assess": ["review", "evaluate", "analyze"],
        "policy": ["plan", "coverage", "contract"],
    }
    expanded = [question]
    for key, synonyms in expansions.items():
        if key in question.lower():
            expanded.extend([question.lower().replace(key, s) for s in synonyms])
    return " OR ".join(expanded)


def ask_question(vectorstore, question: str, k: int = 4) -> str:
    expanded_query = expand_query(question)

    try:
        if hasattr(vectorstore, "max_marginal_relevance_search"):
            relevant_docs = vectorstore.max_marginal_relevance_search(
                expanded_query,
                k=k,
                fetch_k=max(k * 3, 10),
            )
        else:
            relevant_docs = vectorstore.similarity_search(expanded_query, k=k)
    except AssertionError:
        print("MMR search hit a stale index mismatch. Falling back to similarity search.")
        relevant_docs = vectorstore.similarity_search(expanded_query, k=k)

    if not relevant_docs:
        return "I don’t have enough information in the provided documents."

    context_text = "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for doc in relevant_docs
    )

    system_prompt = """You are a domain support assistant.
Answer only using the retrieved context below.
If the context does not contain enough information, reply exactly with:
I don’t have enough information in the provided documents.

Rules:
- Use only the provided document content.
- Do not use outside knowledge.
- Keep the answer concise but specific.
- If a follow-up question depends on earlier conversation, use previous context.
- Mention the relevant source document name when supported by the retrieved text.

Retrieved context:
{context}
"""

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{question}"),
    ])

    response = llm.invoke(prompt.format_messages(context=context_text, question=question))
    answer = response.content.strip()

    conversation_history.append(f"User: {question}")
    conversation_history.append(f"Assistant: {answer}")
    return answer


# Chatbot logic
This cell defines the retrieval prompt and answer logic that uses only the relevant document context and supports follow-up questions.


# Demo run
This cell tests sample questions and starts the interactive chat so the assistant can answer follow-up queries.


In [7]:
# Keep the sample questions for reference, but do not use them automatically.
print("Ask a question about the insurance documents.")

while True:
    user_input = input("\nYou: ").strip()
    if user_input.lower() in {"exit", "quit", "q"}:
        print("Goodbye.")
        break
    if not user_input:
        continue
    print(f"You asked: {user_input}")
    answer = ask_question(vectorstore, user_input)
    print(f"Assistant: {answer}")


Ask a question about the insurance documents.
You asked: what are the types of life insurance available?
Assistant: There are two major types of life insurance: term life and whole life. 

1. **Term Life**: This is the simplest form of life insurance, paying only if death occurs during the policy term, which usually ranges from one to 30 years. It includes two basic types: level term (where the death benefit remains the same throughout the policy) and decreasing term.

2. **Whole Life**: This type of insurance provides coverage for the insured's entire life and typically includes a cash value component. 

(Source: Insurance_Handbook_20103.pdf)
You asked: How to assess my current life insurance policy?
Assistant: To assess your current life insurance policy, you should compare it with any new policy you might consider. Review the benefits of your current policy and determine if you can make changes to better meet your needs. Additionally, consider how any changes in your health may affe